In [2]:
import pandas as pd
import numpy as np

df_a = pd.read_csv('../data/a.csv')
df_d = pd.read_csv('../data/d.csv')
df_p = pd.read_csv('../data/p.csv')

df_a.drop(columns=["bias_label"], inplace=True)
df_d.drop(columns=["bias_label"], inplace=True)
df_p.drop(columns=["bias_label"], inplace=True)

In [3]:
# concatenate dataframes and drop duplicate articles (by article_id)
df = pd.concat([df_a, df_d, df_p], ignore_index=True, sort=False)
df = df.drop_duplicates(subset='article_id', keep='first').reset_index(drop=True)
df.drop(columns=["flags","Unnamed: 8"], inplace=True)

# quick check
print("df shape:", df.shape)
df.head()
df.info()

df shape: (2000, 6)
<class 'pandas.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   article_id    2000 non-null   str  
 1   publisher     2000 non-null   str  
 2   url           2000 non-null   str  
 3   published_at  2000 non-null   str  
 4   title         2000 non-null   str  
 5   body_text     1999 non-null   str  
dtypes: str(6)
memory usage: 93.9 KB


In [4]:
# drop rows with any null values
df.dropna(inplace=True)
print("df shape after dropna:", df.shape)
df.info()

df shape after dropna: (1999, 6)
<class 'pandas.DataFrame'>
Index: 1999 entries, 0 to 1999
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   article_id    1999 non-null   str  
 1   publisher     1999 non-null   str  
 2   url           1999 non-null   str  
 3   published_at  1999 non-null   str  
 4   title         1999 non-null   str  
 5   body_text     1999 non-null   str  
dtypes: str(6)
memory usage: 109.3 KB


In [5]:
#Unicode normalization (NFC - canonical decomposition + composition)
import unicodedata                                                                                                                                          
                                                                                                                                                        
# Unicode normalization (NFC - canonical decomposition + composition)                                                                                       
df['body_text'] = df['body_text'].apply(lambda x: unicodedata.normalize('NFC', x))                                                                        
df['title'] = df['title'].apply(lambda x: unicodedata.normalize('NFC', x))

df.head()


,article_id,publisher,url,published_at,title,body_text
0,f92797eb-a338-5886-a45a-66f01292a912,Lanka Deepa,https://www.lankadeepa.lk/news/දන-තර-මර-අහවන-ක...,2025-09-26 00:00:00+00:00,දැන් තෝරු-මෝරු අහුවෙන කාලේ,දැන් තෝරු-මෝරු අහුවෙන කාලේ\n\nදැන් හාල්මැස්සො ...
1,5c7946aa-a810-59e2-a126-ff58347aad21,BBC Sinhala,https://www.bbc.com/sinhala/articles/c0vy04qd14yo,2024-01-17 00:00:00+00:00,අවුරුදු 67යි. ඇස් දෙකම පේන්නේ නෑ. මම පොල් කඩනවා',අවුරුදු 67යි. ඇස් දෙකම පේන්නේ නෑ. මම පොල් කඩනව...
2,6fc575d1-2be5-5d49-be20-c846617bdacd,Ada,https://www.ada.lk/breaking_news/පිදුරංගල-ගිය-...,2026-02-17 00:00:00+00:00,පිදුරංගල ගිය විදේශිකයා වන අලි ප්‍රහාරයෙන් මරුට,පිදුරංගල ගිය විදේශිකයා වන අලි ප්‍රහාරයෙන් මරුට...
3,47965836-064f-530a-a34d-02f6898fb94a,Ada Derana Sinhala,http://sinhala.adaderana.lk/news/194266,2024-03-07 00:00:00+00:00,හූති ප්‍රහාරයට ලක්වූ නෞකාවේ සිටි ලාංකිකයින් ගැ...,හූති ප්‍රහාරයට ලක්වූ නෞකාවේ සිටි ලාංකිකයින් ගැ...
4,132990f7-a4f4-54ba-8b12-b469d9803bde,Ada Derana Sinhala,http://sinhala.adaderana.lk/news/195779,2024-04-19 00:00:00+00:00,ස්ථාන දෙකකදී ඝාතන දෙකක්,ස්ථාන දෙකකදී ඝාතන දෙකක්\n\nකුලී නිවසක පදිංචිව ...


In [6]:
#Removing title duplication in the body text
def remove_title_from_body(row):                                                                                                                            
    body = row['body_text']                                                                                                                                 
    title = row['title']                                                                                                                                    
    if body.startswith(title):                                                                                                                            
        body = body[len(title):].lstrip('\n').lstrip()
    return body

df['text'] = df.apply(remove_title_from_body, axis=1)

df.info()

<class 'pandas.DataFrame'>
Index: 1999 entries, 0 to 1999
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   article_id    1999 non-null   str  
 1   publisher     1999 non-null   str  
 2   url           1999 non-null   str  
 3   published_at  1999 non-null   str  
 4   title         1999 non-null   str  
 5   body_text     1999 non-null   str  
 6   text          1999 non-null   str  
dtypes: str(7)
memory usage: 124.9 KB


In [7]:
#Whitespace normalization

import re                                                                                                                                                   
                                                                                                                                                            
df['text'] = df['text'].apply(lambda x: re.sub(r'\n{2,}', '\n', x))  # collapse multiple newlines                                                           
df['text'] = df['text'].apply(lambda x: re.sub(r'[ \t]+', ' ', x))   # collapse multiple spaces/tabs                                                      
df['text'] = df['text'].str.strip()                                    # trim leading/trailing whitespace

df.head()

,article_id,publisher,url,published_at,title,body_text,text
0,f92797eb-a338-5886-a45a-66f01292a912,Lanka Deepa,https://www.lankadeepa.lk/news/දන-තර-මර-අහවන-ක...,2025-09-26 00:00:00+00:00,දැන් තෝරු-මෝරු අහුවෙන කාලේ,දැන් තෝරු-මෝරු අහුවෙන කාලේ\n\nදැන් හාල්මැස්සො ...,දැන් හාල්මැස්සො නොව තෝරු මෝරු අහුවෙන කාලය බවමහ...
1,5c7946aa-a810-59e2-a126-ff58347aad21,BBC Sinhala,https://www.bbc.com/sinhala/articles/c0vy04qd14yo,2024-01-17 00:00:00+00:00,අවුරුදු 67යි. ඇස් දෙකම පේන්නේ නෑ. මම පොල් කඩනවා',අවුරුදු 67යි. ඇස් දෙකම පේන්නේ නෑ. මම පොල් කඩනව...,"මෙහි කිසිවක් අඩංගු නැත.Play video, ""දිරිය මිනි..."
2,6fc575d1-2be5-5d49-be20-c846617bdacd,Ada,https://www.ada.lk/breaking_news/පිදුරංගල-ගිය-...,2026-02-17 00:00:00+00:00,පිදුරංගල ගිය විදේශිකයා වන අලි ප්‍රහාරයෙන් මරුට,පිදුරංගල ගිය විදේශිකයා වන අලි ප්‍රහාරයෙන් මරුට...,සිගිරිය පිදුරංගල මාර්ගයේදී ඊයේ සවස වන අලියෙකු ...
3,47965836-064f-530a-a34d-02f6898fb94a,Ada Derana Sinhala,http://sinhala.adaderana.lk/news/194266,2024-03-07 00:00:00+00:00,හූති ප්‍රහාරයට ලක්වූ නෞකාවේ සිටි ලාංකිකයින් ගැ...,හූති ප්‍රහාරයට ලක්වූ නෞකාවේ සිටි ලාංකිකයින් ගැ...,හූති ප්‍රහාරයට ලක්වූ නෞකාවේ සිටි ලාංකිකයින් දෙ...
4,132990f7-a4f4-54ba-8b12-b469d9803bde,Ada Derana Sinhala,http://sinhala.adaderana.lk/news/195779,2024-04-19 00:00:00+00:00,ස්ථාන දෙකකදී ඝාතන දෙකක්,ස්ථාන දෙකකදී ඝාතන දෙකක්\n\nකුලී නිවසක පදිංචිව ...,කුලී නිවසක පදිංචිව සිටි පුද්ගලයෙකුව ඊයේ (18) ර...


In [8]:
from laser_encoders import LaserEncoderPipeline

encoder = LaserEncoderPipeline(lang="sin_Latn")

ValueError: mutable default <class 'fairseq.dataclass.configs.CommonConfig'> for field common is not allowed: use default_factory